# IBM HR Attrition Analytics — All-in-One

This notebook is the single source of truth for the analytical workflow. **Every run reads the CSV supplied to `CSV_SOURCE`**, so the analysis, EDA, model, risk scores, and summary change when a different compatible CSV is supplied.

The full-stack app uses the same normalization and scoring rules through the FastAPI backend. The frontend uploads a CSV to the backend and refreshes its dashboard from the newly loaded dataset.

In [ ]:
from pathlib import Path
import io, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Change this one variable to analyze another CSV. It may be a Path or raw CSV bytes.
CSV_SOURCE = Path('data/IBM HR Employee Attrition Data.csv')
OUTPUT_DIR = Path('outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

REQUIRED = {
    'Attrition', 'Department', 'JobRole', 'OverTime', 'JobSatisfaction',
    'EnvironmentSatisfaction', 'WorkLifeBalance', 'YearsAtCompany',
    'JobLevel', 'StockOptionLevel', 'NumCompaniesWorked', 'DistanceFromHome'
}
ALIASES = {
    'employee number':'EmployeeNumber','employee_number':'EmployeeNumber','employee id':'EmployeeNumber',
    'job role':'JobRole','job_role':'JobRole','department name':'Department','overtime':'OverTime','over time':'OverTime',
    'attrition flag':'Attrition','job satisfaction':'JobSatisfaction','environment satisfaction':'EnvironmentSatisfaction',
    'work life balance':'WorkLifeBalance','years at company':'YearsAtCompany','job level':'JobLevel',
    'stock option level':'StockOptionLevel','number of companies worked':'NumCompaniesWorked',
    'num companies worked':'NumCompaniesWorked','distance from home':'DistanceFromHome',
    'monthly income':'MonthlyIncome','total working years':'TotalWorkingYears',
}

def read_csv_source(source):
    if isinstance(source, (bytes, bytearray)):
        raw = bytes(source)
    else:
        raw = Path(source).read_bytes()
    df = pd.read_csv(io.BytesIO(raw))
    df.columns = [str(c).lstrip('\ufeff').strip() for c in df.columns]
    renamed = {}
    for c in df.columns:
        renamed[c] = ALIASES.get(c.lower().replace('-', ' ').strip(), c)
    df = df.rename(columns=renamed).drop_duplicates().copy()
    missing = sorted(REQUIRED - set(df.columns))
    if missing:
        raise ValueError('CSV is missing required columns: ' + ', '.join(missing))
    for c in df.select_dtypes(include=['object','string']).columns:
        df[c] = df[c].astype('string').str.strip().replace({'': pd.NA}).fillna('Unknown')
    for c in df.columns:
        if c not in df.select_dtypes(include=['object','string']).columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
            if df[c].isna().any():
                median = df[c].median()
                df[c] = df[c].fillna(0 if pd.isna(median) else median)
    df['AttritionFlag'] = (df['Attrition'].astype(str).str.lower() == 'yes').astype(int)
    return df

def available_model_features(df):
    preferred = ['Age','MonthlyIncome','JobLevel','StockOptionLevel','TotalWorkingYears','YearsAtCompany',
                 'YearsInCurrentRole','YearsSinceLastPromotion','YearsWithCurrManager','JobSatisfaction',
                 'EnvironmentSatisfaction','WorkLifeBalance','JobInvolvement','DistanceFromHome',
                 'NumCompaniesWorked','OverTime','BusinessTravel','MaritalStatus','JobRole','Department']
    return [c for c in preferred if c in df.columns]

def train_dynamic_model(df):
    features = available_model_features(df)
    categorical = [c for c in ['OverTime','BusinessTravel','MaritalStatus','JobRole','Department'] if c in features]
    numeric = [c for c in features if c not in categorical]
    X, y = df[features], df['AttritionFlag']
    if y.nunique() < 2 or len(df) < 10:
        # Small/custom files still get deterministic analytical scores; a fitted classifier needs both classes and enough rows.
        prob = (0.15 + 0.7 * (df['AttritionFlag'].astype(float)))
        scored = df.copy(); scored['churn_probability'] = prob; scored['risk_tier'] = pd.cut(prob, [-np.inf,.4,.7,np.inf], labels=['Low','Medium','High'])
        return None, {'records':len(df), 'note':'Classifier skipped because the uploaded file does not contain enough rows/classes.'}, scored
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    pre = ColumnTransformer([('numeric', StandardScaler(), numeric), ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical)])
    model = Pipeline([('preprocessor', pre), ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))])
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:,1]; pred = (p >= .5).astype(int)
    metrics = {'records':len(df),'attrition_count':int(y.sum()),'attrition_rate':round(float(y.mean()*100),2),
               'train_records':len(X_train),'test_records':len(X_test),'accuracy':round(float(accuracy_score(y_test,pred)),4),
               'precision':round(float(precision_score(y_test,pred,zero_division=0)),4),'recall':round(float(recall_score(y_test,pred,zero_division=0)),4),
               'roc_auc':round(float(roc_auc_score(y_test,p)),4),'confusion_matrix':confusion_matrix(y_test,pred).tolist(),
               'features':features,'target_definition':"Attrition == 'Yes'"}
    all_p = model.predict_proba(df[features])[:,1]
    scored = df.copy(); scored['churn_probability'] = all_p; scored['risk_tier'] = pd.cut(all_p, [-np.inf,.4,.7,np.inf], labels=['Low','Medium','High'])
    return model, metrics, scored

def analyze(source=CSV_SOURCE):
    df = read_csv_source(source)
    model, metrics, scored = train_dynamic_model(df)
    summary = {
        'records': len(df),
        'columns': len(df.columns),
        'attrition_rate': round(float(df['AttritionFlag'].mean()*100),2),
        'overtime_attrition_rate': round(float(df.loc[df['OverTime'].astype(str).str.lower()=='yes','AttritionFlag'].mean()*100),2),
        'department_attrition': (df.groupby('Department')['AttritionFlag'].mean().mul(100).sort_values(ascending=False).round(2)).to_dict(),
        'role_attrition': (df.groupby('JobRole')['AttritionFlag'].mean().mul(100).sort_values(ascending=False).round(2)).to_dict(),
        'metrics': metrics,
    }
    scored_path = OUTPUT_DIR / 'employee_risk_scores.csv'
    scored.to_csv(scored_path, index=False)
    (OUTPUT_DIR/'analysis_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
    df.to_csv(OUTPUT_DIR/'cleaned_uploaded_dataset.csv', index=False)
    return df, scored, summary

df, scored, summary = analyze()
print(json.dumps(summary, indent=2))

## Interactive upload

Run this cell in Jupyter/Colab and select **any compatible CSV**. The notebook reruns the complete analysis on that file instead of the bundled sample.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    uploader = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')
    output = widgets.Output()
    def _on_upload(change):
        if not uploader.value: return
        item = next(iter(uploader.value.values())) if isinstance(uploader.value, dict) else uploader.value[0]
        name = item.get('name', 'uploaded.csv'); content = item['content']
        with output:
            clear_output()
            try:
                global df, scored, summary
                df, scored, summary = analyze(content)
                print(f'Analyzed: {name} — {len(df):,} rows')
                print(json.dumps(summary, indent=2))
            except Exception as exc:
                print(f'Upload failed: {exc}')
    uploader.observe(_on_upload, names='value')
    display(uploader, output)
except ImportError:
    print('ipywidgets is not installed; set CSV_SOURCE to another CSV path and rerun the analysis cell.')

## Data quality

In [ ]:
print('Shape:', df.shape)
print('Duplicate rows:', int(df.duplicated().sum()))
print('Missing cells:', int(df.isna().sum().sum()))
display(df.head())

## Dynamic EDA

All charts below are calculated from the currently loaded dataset.

In [ ]:
def plot_attrition_rates(df):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    dept = df.groupby('Department')['AttritionFlag'].mean().mul(100).sort_values()
    axes[0].barh(dept.index, dept.values); axes[0].set_title('Attrition by department'); axes[0].set_xlabel('%')
    ot = df.groupby('OverTime')['AttritionFlag'].mean().mul(100)
    axes[1].bar(ot.index.astype(str), ot.values); axes[1].set_title('Attrition by overtime'); axes[1].set_ylabel('%')
    tenure = df.assign(TenureBand=(df['YearsAtCompany']//2)*2).groupby('TenureBand')['AttritionFlag'].mean().mul(100)
    axes[2].plot(tenure.index, tenure.values, marker='o'); axes[2].set_title('Attrition by tenure band'); axes[2].set_xlabel('Years'); axes[2].set_ylabel('%')
    fig.tight_layout(); plt.show()
plot_attrition_rates(df)

## Model and risk outputs

The classifier is retrained from the currently loaded CSV. If a custom file is too small or has only one target class, the notebook reports that limitation rather than pretending a valid classifier was trained.

In [ ]:
cols = [c for c in ['EmployeeNumber','Age','Department','JobRole','Attrition','OverTime','MonthlyIncome','YearsAtCompany','churn_probability','risk_tier'] if c in scored.columns]
display(scored[cols].head(20))
print('Output files:', OUTPUT_DIR/'cleaned_uploaded_dataset.csv', OUTPUT_DIR/'employee_risk_scores.csv', OUTPUT_DIR/'analysis_summary.json')

## Full-stack contract

The FastAPI backend exposes:
- `GET /api/health`
- `GET /api/dataset`
- `GET /api/employees`
- `GET /api/analysis`
- `POST /api/dataset/upload`
- `POST /api/dataset/reset`
- `GET /api/export`

The React frontend uploads a CSV to `/api/dataset/upload`, then reloads `/api/employees`. Therefore charts, KPIs, filters, and the employee-risk table are rebuilt from the uploaded dataset.